# Lumut Home: Data Cleaning Notebook

**Purpose:** inspect and clean the five raw CSV files so they are ready for analysis.
**Rule:** the raw files are never edited. Every fix happens in code, and every fix is counted and logged.

**Notes**
* Cells run top to bottom. Each section restarts from the untouched raw copy (for example `ads = ads_raw.copy()`), so any section can be re-run safely.
* Every fix is counted and written to a cleaning log, saved at the end as `data_clean/cleaning_log.csv`.
* Validation checks at the end confirm row reconciliation, uniqueness, allowed values, dates and joins.

## Part 1: Setup and loading

In [1]:
import pandas as pd    # pandas: tables (DataFrames) and data cleaning
import numpy as np     # numpy: numbers, NaN (missing values)
import os              # os: folders and file paths

# ---- Folders (change DATA_DIR if your raw files are somewhere else) ----
DATA_DIR = "../data_raw/"        # folder holding the ORIGINAL raw CSV files
OUT_DIR  = "../data_clean/"      # folder where cleaned files will be saved
os.makedirs(OUT_DIR, exist_ok=True)

# Show all columns when printing a table
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

In [2]:
# Load each raw CSV into a DataFrame (a table in memory)
campaigns = pd.read_csv(DATA_DIR + "campaigns.csv")
ads       = pd.read_csv(DATA_DIR + "ad_performance_daily.csv")
orders    = pd.read_csv(DATA_DIR + "orders.csv")
customers = pd.read_csv(DATA_DIR + "customers.csv")
fx        = pd.read_csv(DATA_DIR + "fx_rates_monthly.csv")

# Starting sizes: (rows, columns). Expected:
# (111, 13) (73628, 11) (47669, 12) (30499, 6) (21, 2)
print(campaigns.shape, ads.shape, orders.shape, customers.shape, fx.shape)

(111, 13) (73628, 11) (47669, 12) (30499, 6) (21, 2)


In [3]:
# Untouched copies. Each cleaning section starts from one of these,
# and we use them at the end to prove how many rows each fix changed.
campaigns_raw = campaigns.copy()
ads_raw       = ads.copy()
orders_raw    = orders.copy()
customers_raw = customers.copy()
fx_raw        = fx.copy()

### Two helpers used throughout

* `log(...)` writes one line to the **cleaning log** (file, column, issue, rows affected, action, method, assumption).
  At the end, the log is saved as `cleaning_log.csv`. Read every entry and make sure you agree with the assumption.
* `parse_mixed_dates(...)` converts a column that contains **two date formats** into real dates.
  `2025-01-06` (year first) and `06/01/2025` (day first) are parsed **separately**, because a date like
  `06/01/2025` is ambiguous: pandas could read it as 1 June or 6 January. We know from the data dictionary and from
  the rows around it that it is day/month/year.

In [4]:
# ---------- Cleaning log ----------
log_rows = []

def log(file, column, issue, rows_affected, action, method, assumption):
    # Store one cleaning decision. "action" must be Fix, Flag or Remove.
    log_rows.append({
        "file": file, "column": column, "issue": issue,
        "rows_affected": int(rows_affected), "action": action,
        "method": method, "assumption_and_risk": assumption,
    })

# ---------- Mixed date formats ----------
def parse_mixed_dates(s):
    # Turn a text column with YYYY-MM-DD and DD/MM/YYYY dates into real dates.
    s = s.astype(str).str.strip()
    is_iso = s.str.match(r"^\d{4}-\d{2}-\d{2}$")   # e.g. 2025-01-06
    is_dmy = s.str.match(r"^\d{2}/\d{2}/\d{4}$")    # e.g. 06/01/2025
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[is_iso] = pd.to_datetime(s[is_iso], format="%Y-%m-%d")
    out[is_dmy] = pd.to_datetime(s[is_dmy], format="%d/%m/%Y")
    # Safety: every value must have matched one of the two formats
    assert out.notna().all(), "Some dates matched neither format"
    return out

## Part 2: Cleaning

Order of work: small reference tables first (`fx`, `campaigns`), then `ads`, then `customers`, then `orders`.
Customers come **before** orders because merging duplicate customers means re-pointing their orders.

Three rules for every fix:
* **Fix** what is clearly the same thing written differently.
* **Flag** what you cannot resolve (add a column that marks it, keep the original).
* **Remove** only exact duplicates.

### Step 1: `fx_rates_monthly` (21 rows, one month missing)

**Concepts:** convert text to real dates, compare against what you *expect*, estimate a missing value and mark it as estimated.

In [5]:
fx = fx_raw.copy()

# Convert month from text to real date values so pandas can sort and compare it
fx["month"] = pd.to_datetime(fx["month"])
fx.dtypes

month         datetime64[us]
gbp_to_usd           float64
dtype: object

In [6]:
# Every month start the data SHOULD cover (Jan 2025 to Oct 2026; "MS" = month start)
expected = pd.date_range("2025-01-01", "2026-10-01", freq="MS")
print("expected months:", len(expected))   # 22

# Expected months that are NOT in our table (~ flips True/False)
missing = expected[~expected.isin(fx["month"])]
print("missing:", list(missing.strftime("%Y-%m")))   # ['2026-02']

expected months: 22
missing: ['2026-02']


In [7]:
# Neighbouring rates around the gap
fx[(fx["month"] >= "2025-12-01") & (fx["month"] <= "2026-04-01")]

,month,gbp_to_usd
11,2025-12-01,1.2369
12,2026-01-01,1.2240
13,2026-03-01,1.2226
14,2026-04-01,1.2256


**Decision:** January 2026 is 1.2240 and March 2026 is 1.2226, a gap of only 0.0014 (about 0.1%).
The choice barely changes any converted amount, so we use the **average of the two neighbouring months** and
mark the row as estimated. *Risk:* if the real February rate was unusual, February converted figures are slightly off.

In [8]:
# Rates for the months either side of the gap
jan_rate = fx.loc[fx["month"] == "2026-01-01", "gbp_to_usd"].iloc[0]
mar_rate = fx.loc[fx["month"] == "2026-03-01", "gbp_to_usd"].iloc[0]
avg_rate = round((jan_rate + mar_rate) / 2, 4)

# One-row table for the missing month
new_row = pd.DataFrame({"month": [pd.Timestamp("2026-02-01")], "gbp_to_usd": [avg_rate]})

# Mark real rows as real and the new row as estimated
fx["is_estimated"] = False
new_row["is_estimated"] = True

# Stack, sort by month and renumber
fx = pd.concat([fx, new_row], ignore_index=True).sort_values("month").reset_index(drop=True)

print(fx.shape)                  # (22, 3)
fx[fx["is_estimated"]]           # only Feb 2026

(22, 3)


,month,gbp_to_usd,is_estimated
13,2026-02-01,1.2233,True


In [9]:
log("fx_rates_monthly", "month", "February 2026 missing (21 of 22 months)", 1, "Fix",
    "Added the missing month with the average of the Jan and Mar 2026 rates; added is_estimated=True",
    "Rate moved smoothly between neighbours (they differ by about 0.1%). Risk is small.")

### Step 2: `campaigns` (111 rows)

**Concepts:** `.str.strip()` removes stray spaces; a **mapping dictionary** turns messy labels into official ones.
Checks you should always run after mapping: `.value_counts()` and "did anything become blank?".

In [10]:
campaigns = campaigns_raw.copy()

# Remove leading/trailing spaces from every text column.
# "Paid Social " and "Paid Social" look identical but are different values.
for c in campaigns.columns:
    if not pd.api.types.is_numeric_dtype(campaigns[c]):      # text columns only
        campaigns[c] = campaigns[c].str.strip()

# What distinct channels exist now?
campaigns["channel"].value_counts()

channel
Paid Search        36
Paid Social        32
Shopping           11
Display             7
Video               6
paid social         3
Email               3
Google Shopping     2
email               2
Social              2
Search              2
paid search         1
YouTube Ads         1
display             1
GDN                 1
PPC                 1
Name: count, dtype: int64

In [11]:
# ---- channel: map every variant to one of the 6 official channels ----
# The vague ones (Search, Social, GDN, PPC) were checked against campaign_name and platform:
#   Search -> names say PaidSearch on Google Ads
#   Social -> names say PaidSocial on Meta / TikTok
#   GDN    -> name says Display on Google Display Network
#   PPC    -> name says PaidSearch on Microsoft Ads
channel_map = {
    "paid social": "Paid Social", "paid search": "Paid Search",
    "display": "Display", "email": "Email",
    "Search": "Paid Search", "Social": "Paid Social",
    "Google Shopping": "Shopping", "YouTube Ads": "Video",
    "GDN": "Display", "PPC": "Paid Search",
}

# Safety: no entry may be empty
assert all(v != "" for v in channel_map.values())

campaigns["channel"] = campaigns["channel"].replace(channel_map)

# Check: exactly 6 official values, adding up to 111 rows
print(campaigns["channel"].value_counts())
assert set(campaigns["channel"]) == {"Paid Search","Paid Social","Shopping","Display","Video","Email"}

channel
Paid Search    40
Paid Social    37
Shopping       13
Display         9
Video           7
Email           5
Name: count, dtype: int64


In [12]:
# ---- platform: map every variant to one of the 8 official platforms ----
campaigns["platform"].value_counts()

platform
Google Ads                41
Meta                      24
Microsoft Ads              9
Google Display Network     8
YouTube                    5
Pinterest                  4
Klaviyo                    3
Facebook Ads               3
Meta Ads                   2
klaviyo                    2
pinterest                  2
AdWords                    1
Youtube                    1
TikTok                     1
Google Display             1
google ads                 1
YouTube Ads                1
facebook                   1
Bing Ads                   1
Name: count, dtype: int64

In [13]:
platform_map = {
    "Facebook Ads": "Meta", "Meta Ads": "Meta", "facebook": "Meta",
    "google ads": "Google Ads", "AdWords": "Google Ads",
    "Bing Ads": "Microsoft Ads",
    "Google Display": "Google Display Network",
    "Youtube": "YouTube", "YouTube Ads": "YouTube",
    "klaviyo": "Klaviyo", "pinterest": "Pinterest",
}
assert all(v != "" for v in platform_map.values())

campaigns["platform"] = campaigns["platform"].replace(platform_map)

print(campaigns["platform"].value_counts())
assert set(campaigns["platform"]) == {"Google Ads","Microsoft Ads","Meta","TikTok","Pinterest",
                                      "Google Display Network","YouTube","Klaviyo"}

platform
Google Ads                43
Meta                      30
Microsoft Ads             10
Google Display Network     9
YouTube                    7
Pinterest                  6
Klaviyo                    5
TikTok                     1
Name: count, dtype: int64


In [14]:
# Cross-check the two cleaned columns: each platform should belong to a sensible channel
pd.crosstab(campaigns["platform"], campaigns["channel"])

channel,Display,Email,Paid Search,Paid Social,Shopping,Video
platform,,,,,,
Google Ads,0,0,30,0,13,0
Google Display Network,9,0,0,0,0,0
Klaviyo,0,5,0,0,0,0
Meta,0,0,0,30,0,0
Microsoft Ads,0,0,10,0,0,0
Pinterest,0,0,0,6,0,0
TikTok,0,0,0,1,0,0
YouTube,0,0,0,0,0,7


In [15]:
# How many rows did the two label fixes change? (cleaned vs raw copy)
ch_changed = (campaigns["channel"] != campaigns_raw["channel"]).sum()
pl_changed = (campaigns["platform"] != campaigns_raw["platform"]).sum()
print("channel rows changed:", ch_changed, "| platform rows changed:", pl_changed)

log("campaigns", "channel", "Label variants (e.g. 'paid social', 'PPC', 'GDN', 'Search')", ch_changed, "Fix",
    "Stripped spaces; mapped every variant to one of 6 official channels with a dictionary",
    "Vague labels were resolved using campaign_name and platform as evidence. Risk: low.")
log("campaigns", "platform", "Label variants (e.g. 'Facebook Ads', 'AdWords', 'Bing Ads')", pl_changed, "Fix",
    "Mapped every variant to one of 8 official platforms with a dictionary",
    "Each variant clearly refers to one platform. Risk: low.")

channel rows changed: 16 | platform rows changed: 16


In [16]:
# ---- Dates: convert to real dates, then find end dates that come BEFORE start dates ----
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"]   = pd.to_datetime(campaigns["end_date"])

swapped = campaigns["end_date"] < campaigns["start_date"]
print("end before start:", swapped.sum(), list(campaigns.loc[swapped, "campaign_id"]))
campaigns.loc[swapped, ["campaign_id", "start_date", "end_date"]]

end before start: 3 ['CMP-0012', 'CMP-0043', 'CMP-0068']


,campaign_id,start_date,end_date
11,CMP-0012,2025-07-14,2025-03-02
42,CMP-0043,2026-01-05,2025-09-25
67,CMP-0068,2026-04-07,2026-01-21


In [17]:
# The two dates look swapped. We swap them back and FLAG the rows.
# (In Step 3 we check this against the real first/last ad dates of these campaigns.)
campaigns["dates_swapped"] = swapped
campaigns.loc[swapped, ["start_date", "end_date"]] = (
    campaigns.loc[swapped, ["end_date", "start_date"]].values
)

# Verify: no end date is before its start date any more
assert (campaigns["end_date"] >= campaigns["start_date"]).all()
campaigns.loc[campaigns["dates_swapped"], ["campaign_id", "start_date", "end_date"]]

,campaign_id,start_date,end_date
11,CMP-0012,2025-03-02,2025-07-14
42,CMP-0043,2025-09-25,2026-01-05
67,CMP-0068,2026-01-21,2026-04-07


In [18]:
# ---- Blank budgets, broken names, repeated names: FLAG, do not invent values ----
campaigns["budget_missing"] = campaigns["planned_budget"].isna()

# Names should follow Market_Channel_Type_Theme_YYYYMM, e.g. UK_PaidSocial_Retargeting_NewArrivals_202503
campaigns["name_follows_convention"] = campaigns["campaign_name"].str.match(
    r"^(UK|US)_[A-Za-z]+_[A-Za-z]+_[A-Za-z]+_\d{6}$")

# Same name used by more than one campaign_id
campaigns["name_duplicated"] = campaigns["campaign_name"].duplicated(keep=False)

print("budget blank:", campaigns["budget_missing"].sum())
print("names breaking convention:", (~campaigns["name_follows_convention"]).sum())
print("rows sharing a name:", campaigns["name_duplicated"].sum())

n_swapped = int(swapped.sum())
log("campaigns", "start_date/end_date", "End date earlier than start date", n_swapped, "Fix + Flag",
    "Swapped the two dates and set dates_swapped=True; verified against real ad dates in Step 3",
    "Dates were entered the wrong way round. Risk: low once verified against ad data.")
log("campaigns", "planned_budget", "Blank budget", campaigns["budget_missing"].sum(), "Flag",
    "Left blank; added budget_missing=True", "A budget cannot be invented. Pacing analysis excludes these campaigns.")
log("campaigns", "campaign_name", "Names break the naming convention", (~campaigns["name_follows_convention"]).sum(), "Flag",
    "Added name_follows_convention flag; names unchanged", "Names are for people; campaign_id is the key. Risk: none.")
log("campaigns", "campaign_name", "Same name on different campaign_ids", campaigns["name_duplicated"].sum(), "Flag",
    "Added name_duplicated flag", "Never join on campaign_name, always on campaign_id.")

budget blank: 6
names breaking convention: 8
rows sharing a name: 14


### Step 3: `ad_performance_daily` (73,628 rows)

Order matters here:
1. Clean the key (`campaign_id`) first, because later checks join on it.
2. Standardise dates, `device` and `currency`.
3. Remove exact duplicates **after** standardising (a duplicate can hide behind a different spelling).
4. Separate rows whose `campaign_id` does not exist in `campaigns`.
5. Flag spend, clicks and conversion-value problems (do not silently overwrite).

In [19]:
ads = ads_raw.copy()

# ---- 1. campaign_id: remove spaces and upper-case it ("cmp-0001 " -> "CMP-0001") ----
ads["campaign_id"] = ads["campaign_id"].str.strip().str.upper()
id_changed = (ads["campaign_id"] != ads_raw["campaign_id"]).sum()
print("campaign_id rows changed:", id_changed)

# ---- 2a. Dates: two formats parsed separately ----
non_iso = (~ads_raw["date"].str.match(r"^\d{4}-\d{2}-\d{2}$")).sum()
ads["date"] = parse_mixed_dates(ads["date"])
print("non-ISO dates fixed:", non_iso, "| date range:", ads["date"].min().date(), "to", ads["date"].max().date())

campaign_id rows changed: 220


non-ISO dates fixed: 1473 | date range: 2025-01-01 to 2026-10-31


In [20]:
# ---- 2b. device: unify the case, and label blank/'Unknown' as "Unknown" ----
ads["device_was_blank"] = ads["device"].isna()       # remember which were truly blank

device_map = {"mobile": "Mobile", "desktop": "Desktop", "tablet": "Tablet"}
ads["device"] = ads["device"].str.strip().str.lower().map(device_map).fillna("Unknown")

print(ads["device"].value_counts())
assert set(ads["device"]) == {"Mobile", "Desktop", "Tablet", "Unknown"}

device
Tablet     24014
Mobile     23978
Desktop    23953
Unknown     1683
Name: count, dtype: int64


In [21]:
# ---- 2c. currency: unify to GBP / USD (conversion comes later, in the analysis) ----
currency_map = {"gbp": "GBP", "£": "GBP", "usd": "USD", "$": "USD", "us$": "USD"}
ads["currency"] = ads["currency"].str.strip().str.lower().map(currency_map)
assert ads["currency"].notna().all(), "An unexpected currency label was found"
print(ads["currency"].value_counts())

currency
USD    40806
GBP    32822
Name: count, dtype: int64


In [22]:
# ---- 3. Exact duplicate rows (checked AFTER standardising) ----
n_before = len(ads)
ads = ads.drop_duplicates().reset_index(drop=True)
n_dups = n_before - len(ads)
print("duplicate rows removed:", n_dups, "| rows now:", len(ads))

duplicate rows removed: 584 | rows now: 73044


In [23]:
# ---- 4. Orphans: campaign_ids that do not exist in campaigns ----
valid_ids = set(campaigns["campaign_id"])
is_orphan = ~ads["campaign_id"].isin(valid_ids)

print("orphan IDs:", sorted(ads.loc[is_orphan, "campaign_id"].unique()))
print("orphan rows:", is_orphan.sum(), "| spend in orphan rows:", round(ads.loc[is_orphan, "spend"].sum(), 2))

# Quarantine them in their own table (not deleted silently)
ads_unmapped = ads[is_orphan].copy()
ads = ads[~is_orphan].reset_index(drop=True)
print("rows now:", len(ads))

orphan IDs: ['CMP-9001', 'CMP-9002', 'CMP-9004', 'CMP-9005', 'CMP-9008', 'CMP-9010', 'CMP-9012', 'CMP-9013', 'CMP-9014', 'CMP-9015', 'CMP-9016', 'CMP-9018', 'CMP-9019', 'CMP-9021', 'CMP-9022', 'CMP-9024', 'CMP-9025', 'CMP-9026', 'CMP-9027', 'CMP-9028', 'CMP-9030', 'CMP-9031', 'CMP-9032', 'CMP-9036', 'CMP-9037', 'CMP-9039', 'CMP-9040', 'CMP-9042', 'CMP-9043', 'CMP-9044', 'CMP-9045', 'CMP-9046', 'CMP-9047', 'CMP-9048', 'CMP-9049', 'CMP-9050', 'CMP-9051', 'CMP-9054', 'CMP-9055', 'CMP-9057', 'CMP-9058', 'CMP-9060', 'CMP-9061', 'CMP-9063', 'CMP-9064', 'CMP-9065', 'CMP-9066', 'CMP-9069', 'CMP-9070', 'CMP-9072', 'CMP-9073', 'CMP-9074', 'CMP-9075', 'CMP-9077', 'CMP-9078', 'CMP-9083', 'CMP-9086', 'CMP-9088', 'CMP-9089', 'CMP-9090', 'CMP-9091', 'CMP-9093', 'CMP-9096', 'CMP-9097']
orphan rows: 109 | spend in orphan rows: 2554.09
rows now: 72935


In [24]:
# ---- 5a. spend: flag blank, negative and suspected decimal-shift (x100) values ----
ads["spend_flag"] = "ok"
ads.loc[ads["spend"].isna(), "spend_flag"] = "missing"
ads.loc[ads["spend"] < 0, "spend_flag"] = "negative"

# A decimal-shift error makes a spend ~100x too big, so it is a WHOLE number far above
# the campaign's usual daily spend. Compare each row with its campaign's median spend.
camp_median = ads.groupby("campaign_id")["spend"].transform("median")
ratio = ads["spend"] / camp_median
is_whole = ads["spend"] == ads["spend"].round(0)
suspect = is_whole & (ratio > 8) & (ads["spend"] > 0)
ads.loc[suspect, "spend_flag"] = "suspected_x100"

print(ads["spend_flag"].value_counts())

spend_flag
ok                72464
missing             364
negative             72
suspected_x100       35
Name: count, dtype: int64


In [25]:
# Sanity test of the rule: cost-per-click of the suspected rows BEFORE and AFTER dividing by 100.
# After /100 it should be in the normal range (well under 5).
chk = ads.loc[suspect, ["campaign_id", "spend", "clicks"]].copy()
chk["cpc_now"]  = (chk["spend"] / chk["clicks"]).round(2)
chk["cpc_div100"] = (chk["spend"] / 100 / chk["clicks"]).round(2)
print(chk[["cpc_now", "cpc_div100"]].describe().round(2))

       cpc_now  cpc_div100
count    35.00       35.00
mean     77.87        0.78
std      49.27        0.49
min       2.86        0.03
25%      42.30        0.42
50%      70.59        0.71
75%     101.68        1.02
max     204.83        2.05


In [26]:
# spend_clean = the spend to use in analysis:
#   * suspected_x100 -> divided by 100
#   * missing        -> stays blank (NaN), never filled with 0
#   * negative       -> kept as is (treated as a platform credit, i.e. net spend)
ads["spend_clean"] = ads["spend"]
ads.loc[suspect, "spend_clean"] = ads.loc[suspect, "spend"] / 100

# ---- 5b. clicks greater than impressions is impossible: FLAG ----
ads["clicks_exceed_impressions"] = ads["clicks"] > ads["impressions"]

# ---- 5c. blank conversion_value: FLAG, leave blank ----
ads["conversion_value_missing"] = ads["conversion_value"].isna()

print("clicks > impressions:", ads["clicks_exceed_impressions"].sum())
print("conversion_value blank:", ads["conversion_value_missing"].sum())
print("total spend (raw sum)  :", round(ads["spend"].sum(), 2))
print("total spend (spend_clean):", round(ads["spend_clean"].sum(), 2))

clicks > impressions: 146
conversion_value blank: 219
total spend (raw sum)  : 1701096.5
total spend (spend_clean): 1599781.88


In [27]:
# ---- Checks that link ads back to campaigns ----
# (a) Does each row's currency match its campaign's currency?
camp_currency = campaigns.set_index("campaign_id")["currency"]
mismatch = (ads["currency"] != ads["campaign_id"].map(camp_currency)).sum()
print("currency mismatches with campaigns:", mismatch)

# (b) Evidence for the 3 campaigns whose dates we swapped: their real first/last ad dates
ad_span = ads.groupby("campaign_id")["date"].agg(first_ad="min", last_ad="max")
check = campaigns.loc[campaigns["dates_swapped"], ["campaign_id", "start_date", "end_date"]].merge(
    ad_span, left_on="campaign_id", right_index=True, how="left")
check

currency mismatches with campaigns: 0


,campaign_id,start_date,end_date,first_ad,last_ad
11,CMP-0012,2025-03-02,2025-07-14,2025-03-02,2025-07-14
42,CMP-0043,2025-09-25,2026-01-05,2025-09-25,2026-01-05
67,CMP-0068,2026-01-21,2026-04-07,2026-01-21,2026-04-07


In [28]:
# The swapped campaigns' ad dates should sit inside the corrected start/end dates
inside = ((check["first_ad"] >= check["start_date"]) & (check["last_ad"] <= check["end_date"])).all()
print("swap confirmed by ad data:", inside)

# Log everything done to ads
log("ad_performance_daily", "campaign_id", "Lowercase / spaces in IDs", id_changed, "Fix", "strip() + upper()", "None. Pure formatting.")
log("ad_performance_daily", "date", "Two date formats (YYYY-MM-DD and DD/MM/YYYY)", non_iso, "Fix",
    "Parsed each format separately with explicit formats", "DD/MM/YYYY confirmed by rows such as 17/01/2025 and by row order.")
log("ad_performance_daily", "device", "Case variants, blanks and 'Unknown'",
    (ads_raw["device"].isna() | ~ads_raw["device"].isin(["Mobile","Desktop","Tablet"])).sum(), "Fix",
    "Lower-cased then mapped to Mobile/Desktop/Tablet; blank and 'Unknown' both labelled 'Unknown'; kept device_was_blank",
    "Blank and 'Unknown' are assumed to mean the same thing (device not recorded).")
log("ad_performance_daily", "currency", "Label variants (gbp, £, usd, $, US$)",
    (~ads_raw["currency"].isin(["GBP", "USD"])).sum(),
    "Fix", "Mapped to GBP/USD; verified against campaigns.currency", "Each variant is unambiguous. Amounts are NOT converted here.")
log("ad_performance_daily", "(whole row)", "Exact duplicate rows", n_dups, "Remove", "drop_duplicates() after standardising", "Duplicates would double-count spend.")
log("ad_performance_daily", "campaign_id", "IDs not found in campaigns (CMP-90xx)", len(ads_unmapped), "Flag (quarantine)",
    "Moved to ads_unmapped.csv, not deleted", "Cannot be attributed to a campaign. Spend in these rows is reported separately.")
log("ad_performance_daily", "spend", "Blank spend", (ads["spend_flag"] == "missing").sum(), "Flag", "spend_flag='missing'; spend_clean left blank", "Not filled with 0: that would understate cost.")
log("ad_performance_daily", "spend", "Negative spend", (ads["spend_flag"] == "negative").sum(), "Flag", "spend_flag='negative'; kept in spend_clean", "Treated as platform credits (net spend). Risk: could be data errors.")
log("ad_performance_daily", "spend", "Suspected decimal-shift (x100) values", (ads["spend_flag"] == "suspected_x100").sum(), "Fix + Flag",
    "Whole-number spend > 8x the campaign median, divided by 100 in spend_clean; original kept in spend",
    "Corrected cost-per-click becomes normal. Risk: a genuine large spend could be wrongly shrunk (rare).")
log("ad_performance_daily", "clicks/impressions", "Clicks greater than impressions", ads["clicks_exceed_impressions"].sum(), "Flag",
    "clicks_exceed_impressions=True; values unchanged", "Impossible rows. Exclude from CTR/CVR in analysis.")
log("ad_performance_daily", "conversion_value", "Blank", ads["conversion_value_missing"].sum(), "Flag", "conversion_value_missing=True", "Not filled; revenue/ROAS for these rows is understated.")

swap confirmed by ad data: True


### Step 4: `customers` (30,499 rows)

New concept: **deduplicating records that are the same person**. Two customer IDs sharing one `email_hash` are one
customer. We keep the **lowest customer_id** as the survivor and build a lookup (`customer_map`) so the orders of the
duplicate can be re-pointed in Step 5.

In [29]:
customers = customers_raw.copy()

# ---- country ----
country_map = {"UK": "UK", "US": "US", "United Kingdom": "UK", "USA": "US"}
customers["country"] = customers["country"].str.strip().map(country_map)
assert customers["country"].notna().all()
print(customers["country"].value_counts())

country
US    16746
UK    13753
Name: count, dtype: int64


In [30]:
# ---- crm_acquisition_channel: free-text variants -> official channels; blanks -> "Unknown" ----
crm_map = {
    "paid search": "Paid Search", "paid social": "Paid Social", "shopping": "Shopping",
    "display": "Display", "email": "Email", "video": "Video", "direct / organic": "Direct / Organic",
    "google": "Paid Search", "google shopping": "Shopping", "facebook": "Paid Social", "youtube": "Video",
}
mapped = customers["crm_acquisition_channel"].str.strip().str.lower().map(crm_map)

# Safety: nothing that had a value may become empty
assert not (mapped.isna() & customers["crm_acquisition_channel"].notna()).any()

customers["crm_acquisition_channel"] = mapped.fillna("Unknown")
print(customers["crm_acquisition_channel"].value_counts())

crm_acquisition_channel
Paid Search         11080
Shopping             8164
Paid Social          4147
Direct / Organic     3336
Unknown              2744
Display               651
Email                 197
Video                 180
Name: count, dtype: int64


In [31]:
# ---- signup_date: convert, then flag dates after the end of the data period ----
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
future = customers["signup_date"] > pd.Timestamp("2026-10-31")
print("signup dates after 2026-10-31:", future.sum())

customers["signup_date_invalid"] = future       # flag
customers.loc[future, "signup_date"] = pd.NaT   # the real date is unknown, so we do not guess

signup dates after 2026-10-31: 304


In [32]:
# ---- Duplicate customers: same email_hash on more than one customer_id ----
with_email = customers[customers["email_hash"].notna()]
ids_per_email = with_email.groupby("email_hash")["customer_id"].nunique()
dup_emails = ids_per_email[ids_per_email > 1].index
print("emails shared by several customer_ids:", len(dup_emails))

dups = with_email[with_email["email_hash"].isin(dup_emails)]
survivor = dups.groupby("email_hash")["customer_id"].transform("min")   # lowest ID survives

# Lookup: duplicate id -> surviving id
customer_map = {old: new for old, new in zip(dups["customer_id"], survivor) if old != new}
print("customer records merged away:", len(customer_map))

customers = customers[~customers["customer_id"].isin(customer_map.keys())].reset_index(drop=True)
print("customers now:", len(customers))

emails shared by several customer_ids: 357


customer records merged away: 357


customers now: 30142


In [33]:
log("customers", "country", "Label variants (United Kingdom, USA)", (customers_raw["country"].isin(["United Kingdom","USA"])).sum(), "Fix",
    "Mapped to UK / US", "Unambiguous.")
log("customers", "crm_acquisition_channel", "Free-text variants and blanks", (customers_raw["crm_acquisition_channel"].isna()).sum() +
    customers_raw["crm_acquisition_channel"].isin(["google","google shopping","facebook","display","email","youtube"]).sum(), "Fix",
    "Mapped variants to official channels; blanks labelled 'Unknown'",
    "'google' is assumed to be Paid Search (it could be Shopping). Risk: small mis-assignment.")
log("customers", "signup_date", "Dates after the end of the data period (2027-01-01)", future.sum(), "Flag",
    "Set to blank and added signup_date_invalid=True", "True date unknown, so not guessed.")
log("customers", "email_hash", "Same email on several customer_ids", len(customer_map), "Remove (merge)",
    "Kept lowest customer_id per email_hash; orders re-pointed using customer_map",
    "Same email_hash means same person. Customers with a blank email_hash cannot be matched.")

### Step 5: `orders` (47,669 rows)

In [34]:
orders = orders_raw.copy()

# Safety: the file must have the 12 original columns (no stray 'Unnamed' column)
assert orders.shape[1] == 12 and not any(c.startswith("Unnamed") for c in orders.columns)

# ---- order_date: two formats ----
od_non_iso = (~orders_raw["order_date"].str.match(r"^\d{4}-\d{2}-\d{2}$")).sum()
orders["order_date"] = parse_mixed_dates(orders["order_date"])
print("non-ISO dates fixed:", od_non_iso, "| range:", orders["order_date"].min().date(), "to", orders["order_date"].max().date())

non-ISO dates fixed: 716 | range: 2025-01-01 to 2026-10-31


In [35]:
# ---- country: UK / US ----
order_country_map = {"uk": "UK", "gb": "UK", "u.k.": "UK", "united kingdom": "UK",
                     "us": "US", "usa": "US", "united states": "US"}
orders["country"] = orders["country"].str.strip().str.lower().map(order_country_map)
assert orders["country"].notna().all()

# Cross-check: GBP orders should be UK and USD orders should be US
print(pd.crosstab(orders["country"], orders["currency"]))

currency    GBP    USD
country               
UK        22818      0
US            0  24851


In [36]:
# ---- is_refunded: 8 different spellings -> True / False ----
refund_map = {"Y": True, "YES": True, "TRUE": True, "1": True,
              "N": False, "NO": False, "FALSE": False, "0": False}
orders["is_refunded"] = orders["is_refunded"].astype(str).str.strip().str.upper().map(refund_map)
assert orders["is_refunded"].notna().all()
orders["is_refunded"] = orders["is_refunded"].astype(bool)
print(orders["is_refunded"].value_counts())

is_refunded
False    44981
True      2688
Name: count, dtype: int64


In [37]:
# ---- Exact duplicate rows ----
n_before = len(orders)
orders = orders.drop_duplicates().reset_index(drop=True)
n_dup_orders = n_before - len(orders)
print("duplicate orders removed:", n_dup_orders, "| rows now:", len(orders))
assert orders["order_id"].is_unique

duplicate orders removed: 284 | rows now: 47385


In [38]:
# ---- Re-point orders of merged duplicate customers to the surviving customer_id ----
before_ids = orders["customer_id"].copy()
orders["customer_id"] = orders["customer_id"].replace(customer_map)
# Count only rows that HAD a customer_id and now have a different one
repointed = (before_ids.notna() & (before_ids != orders["customer_id"])).sum()
print("orders re-pointed to the surviving customer:", repointed)

# ---- Flags (values are NOT changed) ----
orders["order_value_flag"] = "ok"
orders.loc[orders["order_value"] == 0, "order_value_flag"] = "zero"
orders.loc[orders["order_value"] < 0,  "order_value_flag"] = "negative"
orders["customer_missing"] = orders["customer_id"].isna()      # real orders, but no customer to attach
orders["is_direct"] = orders["last_touch_campaign_id"].isna()  # blank touch = direct / unattributed (not an error)

print(orders["order_value_flag"].value_counts())
print("orders without customer_id:", orders["customer_missing"].sum())
print("shipping_cost blank (kept blank, free shipping is 0):", orders["shipping_cost"].isna().sum())
print("direct / unattributed orders:", orders["is_direct"].sum())

orders re-pointed to the surviving customer: 523
order_value_flag
ok          47291
zero           52
negative       42
Name: count, dtype: int64
orders without customer_id: 189
shipping_cost blank (kept blank, free shipping is 0): 1421
direct / unattributed orders: 7897


In [39]:
log("orders", "order_date", "Two date formats", od_non_iso, "Fix", "Parsed each format separately", "DD/MM/YYYY, same as ads.")
log("orders", "country", "Label variants (GB, U.K., USA, ...)", (~orders_raw["country"].isin(["UK","US"])).sum(), "Fix",
    "Mapped to UK / US; cross-checked against currency", "Verified: GBP orders = UK, USD orders = US.")
log("orders", "is_refunded", "8 different encodings", (~orders_raw["is_refunded"].isin(["Y","N"])).sum(), "Fix",
    "Mapped all to True/False", "Y/Yes/TRUE/1 = refunded; N/No/FALSE/0 = not refunded.")
log("orders", "order_id", "Duplicate rows (same order_id)", n_dup_orders, "Remove", "drop_duplicates()", "Exact copies; would double-count revenue.")
log("orders", "customer_id", "Orders of merged duplicate customers", repointed, "Fix", "Re-pointed using customer_map", "Same person (shared email_hash).")
log("orders", "order_value", "Zero or negative order value", (orders["order_value_flag"] != "ok").sum(), "Flag",
    "order_value_flag = zero / negative; values unchanged", "Could be errors, test orders or adjustments. Exclude from revenue analysis.")
log("orders", "customer_id", "Blank customer_id", orders["customer_missing"].sum(), "Flag", "customer_missing=True", "Real orders; excluded only from customer-level analysis.")
log("orders", "shipping_cost", "Blank shipping_cost", orders["shipping_cost"].isna().sum(), "Flag (left blank)", "Left blank", "Blank is not 0: free shipping is already stored as 0.")
log("orders", "first/last_touch_campaign_id", "Blank touch columns", orders["is_direct"].sum(), "Keep", "Added is_direct for last touch", "Blank = no marketing touch recorded. Real information, not an error.")

## Part 3: Validation

If a check fails, the cleaning is not finished. Read the failing line, go back to that section, and fix it.

In [40]:
# ---- 1. Row reconciliation ----
recon = pd.DataFrame({
    "file":   ["campaigns", "ad_performance_daily", "orders", "customers", "fx_rates_monthly"],
    "before": [len(campaigns_raw), len(ads_raw), len(orders_raw), len(customers_raw), len(fx_raw)],
    "after":  [len(campaigns), len(ads), len(orders), len(customers), len(fx)],
})
recon["difference"] = recon["after"] - recon["before"]
recon

,file,before,after,difference
0,campaigns,111,111,0
1,ad_performance_daily,73628,72935,-693
2,orders,47669,47385,-284
3,customers,30499,30142,-357
4,fx_rates_monthly,21,22,1


In [41]:
# Explain every difference. Ads: duplicates removed + orphans moved out. Orders: duplicates removed.
# Customers: duplicates merged. fx: 1 month added.
assert len(ads) == len(ads_raw) - n_dups - len(ads_unmapped)
assert len(orders) == len(orders_raw) - n_dup_orders
assert len(customers) == len(customers_raw) - len(customer_map)
assert len(fx) == len(fx_raw) + 1
print("Row reconciliation passed")

Row reconciliation passed


In [42]:
# ---- 2. Uniqueness ----
assert campaigns["campaign_id"].is_unique
assert customers["customer_id"].is_unique
assert orders["order_id"].is_unique
assert not ads.duplicated().any()
print("Uniqueness passed")

# ---- 3. Only official values ----
assert set(ads["device"])  <= {"Mobile", "Desktop", "Tablet", "Unknown"}
assert set(ads["currency"]) == {"GBP", "USD"}
assert set(orders["country"]) == {"UK", "US"} and set(customers["country"]) == {"UK", "US"}
assert set(orders["is_refunded"]) == {True, False}
print("Allowed-values check passed")

Uniqueness passed
Allowed-values check passed


In [43]:
# ---- 4. Dates ----
assert ads["date"].between("2025-01-01", "2026-10-31").all()
assert orders["order_date"].between("2025-01-01", "2026-10-31").all()
assert (campaigns["end_date"] >= campaigns["start_date"]).all()
assert customers["signup_date"].dropna().max() <= pd.Timestamp("2026-10-31")
print("Date checks passed")

# ---- 5. Joins ----
assert ads["campaign_id"].isin(campaigns["campaign_id"]).all()
assert orders["last_touch_campaign_id"].dropna().isin(campaigns["campaign_id"]).all()
assert orders["first_touch_campaign_id"].dropna().isin(campaigns["campaign_id"]).all()
assert orders["customer_id"].dropna().isin(customers["customer_id"]).all()
print("Join checks passed")

# ---- 6. Every month in ads/orders has an FX rate ----
months_needed = set(ads["date"].dt.to_period("M").dt.to_timestamp()) | set(orders["order_date"].dt.to_period("M").dt.to_timestamp())
assert months_needed <= set(fx["month"])
print("FX coverage passed")

Date checks passed
Join checks passed


FX coverage passed


In [44]:
# ---- 7. Raw files untouched (sizes match the originals) ----
original_sizes = {"campaigns.csv": 16440, "ad_performance_daily.csv": 5507884,
                  "orders.csv": 3846389, "customers.csv": 1728537, "fx_rates_monthly.csv": 391}
for name, size in original_sizes.items():
    assert os.path.getsize(DATA_DIR + name) == size, name + " was modified!"
print("Raw files untouched")

Raw files untouched


## Part 4: Save the clean data and the cleaning log

The cleaned files go to `data_clean/`. Dates are saved as `YYYY-MM-DD` so Power BI and SQL read them correctly.

In [45]:
fx.to_csv(OUT_DIR + "fx_rates_monthly_clean.csv", index=False)
campaigns.to_csv(OUT_DIR + "campaigns_clean.csv", index=False)
ads.to_csv(OUT_DIR + "ad_performance_daily_clean.csv", index=False)
ads_unmapped.to_csv(OUT_DIR + "ad_performance_unmapped.csv", index=False)
customers.to_csv(OUT_DIR + "customers_clean.csv", index=False)
orders.to_csv(OUT_DIR + "orders_clean.csv", index=False)
pd.DataFrame({"old_customer_id": list(customer_map.keys()),
              "surviving_customer_id": list(customer_map.values())}).to_csv(OUT_DIR + "customer_id_map.csv", index=False)

cleaning_log = pd.DataFrame(log_rows)
cleaning_log.insert(0, "#", range(1, len(cleaning_log) + 1))
cleaning_log.to_csv(OUT_DIR + "cleaning_log.csv", index=False)

print("Saved:", sorted(os.listdir(OUT_DIR)))
cleaning_log

Saved: ['ad_performance_daily_clean.csv', 'ad_performance_unmapped.csv', 'campaigns_clean.csv', 'cleaning_log.csv', 'customer_id_map.csv', 'customers_clean.csv', 'fx_rates_monthly_clean.csv', 'orders_clean.csv']


,#,file,column,issue,rows_affected,action,method,assumption_and_risk
0,1,fx_rates_monthly,month,February 2026 missing (21 of 22 months),1,Fix,Added the missing month with the average of th...,Rate moved smoothly between neighbours (they d...
1,2,campaigns,channel,"Label variants (e.g. 'paid social', 'PPC', 'GD...",16,Fix,Stripped spaces; mapped every variant to one o...,Vague labels were resolved using campaign_name...
2,3,campaigns,platform,"Label variants (e.g. 'Facebook Ads', 'AdWords'...",16,Fix,Mapped every variant to one of 8 official plat...,Each variant clearly refers to one platform. R...
3,4,campaigns,start_date/end_date,End date earlier than start date,3,Fix + Flag,Swapped the two dates and set dates_swapped=Tr...,Dates were entered the wrong way round. Risk: ...
4,5,campaigns,planned_budget,Blank budget,6,Flag,Left blank; added budget_missing=True,A budget cannot be invented. Pacing analysis e...
5,6,campaigns,campaign_name,Names break the naming convention,8,Flag,Added name_follows_convention flag; names unch...,Names are for people; campaign_id is the key. ...
6,7,campaigns,campaign_name,Same name on different campaign_ids,14,Flag,Added name_duplicated flag,"Never join on campaign_name, always on campaig..."
7,8,ad_performance_daily,campaign_id,Lowercase / spaces in IDs,220,Fix,strip() + upper(),None. Pure formatting.
8,9,ad_performance_daily,date,Two date formats (YYYY-MM-DD and DD/MM/YYYY),1473,Fix,Parsed each format separately with explicit fo...,DD/MM/YYYY confirmed by rows such as 17/01/202...
9,10,ad_performance_daily,device,"Case variants, blanks and 'Unknown'",5994,Fix,Lower-cased then mapped to Mobile/Desktop/Tabl...,Blank and 'Unknown' are assumed to mean the sa...


## Output

Cleaned files and the cleaning log are in `data_clean/`. Next step: analysis in SQL and Python, using these files (see `sql/`).